# 19 - Pass 10 Q1: does anything add on top of MLM pretraining? (one commit, GPU)

Same structure as notebooks 15 and 18: fetch the code, run the tests, find the encoder, train two runs at a
time, then **the analysis runs inside this notebook**.

**What it asks** (vault: Experiment Plan, "Pass 10 plan"): MLM task-adaptive pretraining (A20) is the biggest
gain so far. Pass 9 stacked helper heads on L (H), FGM (F) and self-training (S) on plain BERT. This notebook
repeats that 2x2x2 **on the MLM encoder**:

| Arm | H | F | S | Exists already? |
|---|---|---|---|---|
| A20 | | | | Pass 8a |
| A21 | | yes | | Pass 8a |
| **A31** | yes | | | **new** |
| **A32** | | | yes | **new** |
| **A33** | yes | yes | | **new** |
| **A34** | yes | | yes | **new** |
| **A35** | | yes | yes | **new** |
| **A36** | yes | yes | yes | **new** |

Frozen settings (helper weight 0.3, FGM epsilon 0.5, pseudo-label threshold 0.9). The self-training teacher
also starts from the MLM encoder. 5 seeds, 1% and 10% labels, masked test sets scored: 60 runs, plus A20 and
A21 at 1% seed 0 again (must equal Pass 8a exactly). **Nothing is picked on test.** About 1.5-2 hours.

**Before you run, in the right-hand panel:**
- **Input 1:** the same `gainsplit-processed` version as notebooks 10-18.
- **Input 2:** the output of **notebook 15** (Add Input -> Your Work -> Notebooks -> `15-pass8-mlm`, the
  version that made `results/pass8_mlm/`). It holds the seed-0 MLM encoder (`ckpt/mlm`).
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**; you can close the browser.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
The repo's tests train a tiny random BERT for a few steps on made-up data. For Pass 10 they also train
the six new arms from a tiny pretrained encoder, check the self-training teacher starts from that encoder,
check the encoder finder refuses an encoder whose weights differ from the receipt, and run the analysis
on stand-in runs. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Find the MLM encoder from notebook 15 and check it is the same one
Every Pass 10 comparison is against Pass 8a's A20/A21 runs, so the new runs must start from **exactly the
encoder those runs used**. This cell looks for it in the attached notebook-15 output and compares its weight
hashes with the receipt committed in `results/pass8_mlm/pretrain_info.json`. If it is missing or different,
the cell stops: attach the right input (see the top of this notebook) rather than pretraining again.

In [ ]:
from gainsplit.ssl_pretrain import find_encoder
receipt = json.loads(Path(f"{REPO_DIR}/results/pass8_mlm/pretrain_info.json").read_text())
ENC = str(find_encoder("/kaggle/input", "mlm", seed=0, steps=10000, expected_sha256=receipt["weights_sha256"]))
print("MLM encoder (seed 0, 10,000 steps):", ENC, "- weights match the Pass 8a receipt")

## 4. Check the data on the CPU (pre-flight) and list the runs
Checks every file against `manifest.json` and `manifest_pass3.json` (the masked test sets).
Longest first: the self-training + FGM arms at 10% (two models each, FGM doubles the work per step),
then the 1% runs, then the two re-runs.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass10_stack"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN](ENC)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))
for c in configs:
    print(f"{c.run_id:18s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} helper_w={c.helper_weight:g} eps={c.fgm_epsilon:g}")
print(len(configs), "runs")

## Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`.
If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## Checks and analysis (CPU, about 5 minutes)
1. **Re-run check:** A20 and A21 (1%, seed 0) must equal Pass 8a exactly (same encoder, same code).
2. **Plain BERT vs MLM, cell by cell** (`on_bert_vs_mlm`): each cell of the 2x2x2 on both encoders.
3. **Additivity on MLM** (reading aid): each combination's gain over A20 next to the sum of its single gains.
4. **Paired bootstrap** (10,000 resamples, mean over 5 seeds), Holm within each table: `stack_f1` (what each
   ingredient adds on top of MLM, and MLM under the full stack) and `stack_interactions` (do they add up on
   MLM; and does pretraining shrink each ingredient's gain: negative = it shrinks).
5. **Self-training health:** teacher recipe and dev F1, pseudo-labels kept, and **how many of them are right**
   (scored here against the training labels; the training code never reads them).

In [ ]:
from gainsplit import analysis_pass10 as a10
import pandas as pd
pd.set_option("display.width", 200)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass2a", "pass2b", "pass7", "pass8_mlm", "pass9")]
                + [OUT_DIR] if Path(d).exists()]
print("results folders:", RESULTS_DIRS)
train_labels = pd.read_csv(Path(DATA_DIR) / "headlines_train.csv")[["id", "label"]]
tables = a10.run_all(RESULTS_DIRS, "stack", f"{OUT_DIR}/analysis", train_labels=train_labels)
for name, t in tables.items():
    if name != "verdicts":
        print(f"\n=== {name}")
        display(t)
rr = tables.get("rerun_check")
rr10 = None if rr is None else rr[rr["arm"].isin(["A20", "A21"])]
print("re-run check:", "PASS" if rr10 is not None and len(rr10) == 2 and rr10["ok"].all()
      else "DIFFERENCE or missing - report this first")

## Verdict against the predictions written before running
Each prediction counts as **supported** only if the result goes the predicted way **and** passes Holm
(p_holm < 0.05); "no difference" predictions are supported when p_holm >= 0.05 (which is not proof of
equality: read the interval). Predictions and reasons: vault, Experiment Plan, "Pass 10 plan".
Paste this cell's output when you report.

In [ ]:
v = tables["verdicts"]
for _, r in v.iterrows():
    if pd.isna(r.get("mean_diff")):
        print(f"{r.hypothesis} {r.contrast} at {r.budget}%: {r.verdict}")
        continue
    print(f"{r.hypothesis} {r.contrast:44s} {r.budget:3d}%  {r.mean_diff:+.2f} points [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  -> {r.verdict}   ({r.claim})")

## Download
Zips the results folder (runs, logs, receipts, analysis tables). A commit keeps it in the Output tab:
download `results_<plan>.zip` and send it with the notebook log.

In [ ]:
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")